In [ ]:
%matplotlib inline

import os, random, time, sys, warnings

import numpy as np
import scipy as sp
import pandas as pd
import torch
from src.utils import get_weight_masks, get_weight_masks_schaefer, get_file_str

# import plotting libraries
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 10})
plt.rcParams["svg.fonttype"] = "none"
plt.rc('font', family='DejaVu Sans')
import seaborn as sns
sns.set_style("white")
from src.plotting import my_reg_plot

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

In [ ]:
save_figs = True

In [ ]:
# directories
datadir = '/home/lindenmp/research_projects/neuro_rnn/data'
modeldir = '/media/lindenmp/storage_ssd/research_projects/neuro_rnn/results/pytorch/model'
# modeldir = '/media/lindenmp/storage_ssd/research_projects/neuro_rnn/results/pytorch/archive/model'
outdir = '/home/lindenmp/research_projects/neuro_rnn/results/figs'

# data parameters
dt = 100
batch_size = 32
seq_len_multi = 5

# RNN model and training parameters
rnn_model = 'rnn-tanh'
n_runs = 25
n_epochs = 30000
lr = 0.001

# regularization parameters
reg_type = 'l2'
mask_weights = True

In [ ]:
config = {
    'datadir': datadir, 'outdir': outdir,
    'dt': dt, 'batch_size': batch_size,  # data parameters
    'rnn_model': rnn_model, 'n_runs': n_runs, 'n_epochs': n_epochs, 'lr': lr, 'mask_weights': mask_weights,  # RNN model and training parameters
    'reg_type': reg_type, # regularization parameters
}

# Model performance

In [ ]:
color_palette = sns.color_palette("Set2")
kernel_types = ['sa_axis', 'euclidean', None]
kernel_labels = ['RNN-SA', 'RNN-E', 'RNN-Standard']

tasks = ['PerceptualDecisionMaking-v0', 'MultiSensoryIntegration-v0', 'ContextDecisionMaking-v0']
# tasks = ['PerceptualDecisionMaking-v0', 'MultiSensoryIntegration-v0']
n_tasks = len(tasks)

In [ ]:
# for reg_weight in [0.0001, 0.001, 0.0015, 0.002]:
for reg_weight in [0.001, 0.0015, 0.002]:
    config['reg_weight'] = reg_weight
    for hidden_size in [100,]:
        if mask_weights and hidden_size == 50:
            n_io = '9-13'
        elif mask_weights and hidden_size == 100:
            n_io = '14-27'
        elif mask_weights and hidden_size == 200:
            n_io = '31-52'
        else:
            n_io = 'na'
        config['hidden_size'] = hidden_size
        config['n_io'] = n_io
        for decision in [400, 300, 200]:
            config['decision'] = decision
            timing = {'decision': decision}
            env_kwargs = {'dt': dt, 'timing': timing}
            config['env_kwargs'] = env_kwargs

            f, ax = plt.subplots(1, n_tasks, figsize=(8.5, 3))
            # f, ax = plt.subplots(n_tasks, 1, figsize=(8.5, 5))

            for i, task in enumerate(tasks):
                if task == 'PerceptualDecisionMaking-v0':
                    seq_len = 22
                elif task == 'MultiSensoryIntegration-v0':
                    seq_len = 11
                elif task == 'ContextDecisionMaking-v0':
                    seq_len = 13

                seq_len = seq_len + int((decision - 100) / dt)
                seq_len = seq_len * seq_len_multi

                config['task'] = task
                config['seq_len'] = seq_len

                for j, kernel_type in enumerate(kernel_types):
                    config['kernel_type'] = kernel_type

                    # load data
                    file_str = get_file_str(config)
                    log_args = np.load(os.path.join(modeldir, file_str + '.npy'), allow_pickle=True).item()
                    print(file_str)
                    # test_accuracy = log_args['test_accuracy'][:, 1:] * 100
                    # test_accuracy = log_args['test_accuracy'][:, 1:151] * 100
                    test_accuracy = log_args['test_accuracy'][:, 1:201] * 100

                    n_epochs_actual = test_accuracy.shape[1] * 100
                    n_logged_epochs = int(n_epochs_actual / 100)
                    x_step = int(n_epochs_actual / (n_logged_epochs))
                    x = np.arange(x_step, n_epochs_actual + x_step, x_step)
                    x2 = np.arange(1, n_epochs_actual)

                    # compute mean and ci
                    accuracy_mean = test_accuracy.mean(axis=0)
                    accuracy_std = test_accuracy.std(axis=0)
                    ci = 1.96 * (accuracy_std / np.sqrt(test_accuracy.shape[0]))
                    ci_lower = accuracy_mean - ci
                    ci_upper = accuracy_mean + ci

                    ax[i].plot(x, accuracy_mean, color=color_palette[j], label=str(kernel_labels[j]))
                    ax[i].fill_between(x, ci_lower, ci_upper, color=color_palette[j], alpha=0.15)

                ax[i].set_xlabel('Epochs')
                ax[i].set_ylabel('Test Accuracy (%)')
                ax[i].set_ylim([-5, 105])
                # ax[i].set_ylim([50, 105])
                if i == 0:
                    ax[i].legend(loc='lower right')
                ax[i].set_title(task)

            sns.despine(offset=10, trim=True, left=False, right=True, top=True, bottom=False)

            # file_str = '-'.join(file_str.split('-')[:-1])
            # f.suptitle(file_str)
            f.tight_layout()
            plt.show()

            if save_figs:
                try:
                    file_str = 'accuracy_r{:}_d{:}'.format(str(reg_weight).split('.')[1], decision)
                except:
                    file_str = 'accuracy_r{:}_d{:}'.format(reg_weight, decision)
                f.savefig(os.path.join(outdir, file_str), dpi=300, bbox_inches='tight', pad_inches=0.01)